# NB19: Publication-Ready Artifacts

Generates the architecture diagram, LaTeX ablation table, and all figures for the Phase 3 IEEE report.

In [1]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json, os

DIAG_DIR    = "../docs/diagrams"
RESULTS_DIR = "../data/results"
os.makedirs(DIAG_DIR, exist_ok=True)
print("Environment ready.")


Environment ready.


In [2]:
# ── Architecture diagram ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlim(0, 15); ax.set_ylim(0, 10.5); ax.axis('off')
fig.patch.set_facecolor('#F8F9FA')

def box(x, y, w, h, title, sub="", fc="#4A90D9", tc="white", fs=10, style="round,pad=0.15"):
    r = mpatches.FancyBboxPatch((x-w/2,y-h/2), w, h, boxstyle=style,
                                facecolor=fc, edgecolor="#1A1A2E", linewidth=1.8, zorder=3)
    ax.add_patch(r)
    dy = 0.18 if sub else 0
    ax.text(x, y+dy, title, ha='center', va='center', fontsize=fs, fontweight='bold',
            color=tc, zorder=4)
    if sub:
        ax.text(x, y-dy, sub, ha='center', va='center', fontsize=7.5,
                color=tc, style='italic', zorder=4)

def arr(x1,y1,x2,y2,lbl="",c="#333333"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="-|>",color=c,lw=1.8), zorder=2)
    if lbl:
        mx,my=(x1+x2)/2,(y1+y2)/2
        ax.text(mx+0.12,my,lbl,fontsize=7.5,color="#555555",style='italic',zorder=5)

# Nodes
box(7.5, 10.0, 4.0, 0.75, "Input Sequence", "(N, 20, 90) — 20-day window x 90 features", fc="#455A64")
box(4.5,  7.8, 4.2, 1.3,  "RATAN-CE",
    "Regime-Gated Conv + Cross-Asset Attn\n(CE loss, class-balanced)", fc="#1565C0", fs=11)
box(2.0,  5.2, 3.2, 0.85, "Feature Attention alpha", "(N, 90)  softmax weights", fc="#1976D2", fs=9)
box(6.2,  5.2, 3.0, 0.85, "RATAN Probs P_R",     "(N, 3)  P(sell/hold/buy)",  fc="#1976D2", fs=9)
box(2.0,  3.3, 3.2, 0.85, "Attention Scaling",   "X_scaled = X[:, :90] * alpha", fc="#0288D1", fs=9)
box(5.8,  3.3, 3.8, 1.1,  "LightGBM (Augmented)",
    "Input: (N, 93) = 90 scaled + 3 P_R\nClass-balanced, early-stopping", fc="#2E7D32", fs=10)
box(5.8,  1.5, 3.0, 0.85, "LGBM Probs P_L",      "(N, 3)  P(sell/hold/buy)", fc="#388E3C", fs=9)
box(11.5, 4.2, 3.5, 1.5,  "Confidence Gating MLP",
    "[P_R||P_L||regime||vol||agree]\n-> (N, 9)  ->  label + conf",
    fc="#E65100", fs=9.5)
box(11.5, 1.5, 3.2, 0.85, "Final Prediction",
    "{sell, hold, buy}  if conf >= theta  else abstain", fc="#BF360C", fs=9)

# Arrows
arr(7.5, 9.62, 4.5, 8.45,  "")
arr(2.8, 7.15, 2.0, 5.62,  "alpha (N,90)", "#1565C0")
arr(5.6, 7.15, 6.2, 5.62,  "P_R (N,3)","#1565C0")
arr(2.0, 4.78, 2.0, 3.75,  "")
arr(3.6, 3.3,  3.9, 3.3,   "")
arr(6.2, 4.78, 5.8, 3.85,  "P_R augment")
arr(5.8, 2.75, 5.8, 1.92,  "")
arr(6.5, 5.2, 10.0, 4.7,   "P_R (N,3)", "#1565C0")
arr(7.3, 1.5,  9.8, 3.0,   "P_L (N,3)", "#2E7D32")
arr(11.5,3.45, 11.5,1.92,  "")

# ── Legend
legend = [
    mpatches.Patch(color="#1565C0", label="DL Component (RATAN-CE)"),
    mpatches.Patch(color="#2E7D32", label="ML Component (LightGBM-aug)"),
    mpatches.Patch(color="#E65100", label="Fusion / Gating Layer"),
    mpatches.Patch(color="#455A64", label="Input Data"),
]
ax.legend(handles=legend, loc="lower left", fontsize=10, framealpha=0.95,
          edgecolor="#CCC", fancybox=True)

ax.set_title("RLSH: RATAN-LightGBM Synergistic Hybrid Architecture",
             fontsize=15, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig(f"{DIAG_DIR}/rlsh_architecture.pdf", bbox_inches='tight', dpi=300)
plt.savefig(f"{DIAG_DIR}/rlsh_architecture.png", bbox_inches='tight', dpi=150)
plt.close()
print("Architecture diagram saved.")


Architecture diagram saved.


In [3]:
# ── LaTeX ablation table ──────────────────────────────────────────────────────
with open(f"{RESULTS_DIR}/ablation_table.json") as f:
    abl = json.load(f)

B = "\\"
lines = [
    B + "begin{table}[ht]",
    B + "centering",
    B + "caption{Ablation Study: Component Contribution to RLSH Performance}",
    B + "label{tab:ablation}",
    B + "begin{tabular}{lccc}",
    B + "toprule",
    B + "textbf{Model} & " + B + "textbf{Accuracy} & " + B + "textbf{Dir. Acc.} & " + B + "textbf{Coverage} " + B + B,
    B + "midrule",
]
for r in abl["rows"]:
    lines.append(f"{r[0]} & {r[1]} & {r[2]} & {r[3]} " + B + B)
lines += [B + "bottomrule", B + "end{tabular}", B + "end{table}"]
latex = "\n".join(lines)
with open(f"{DIAG_DIR}/ablation_table.tex", "w") as f:
    f.write(latex)
print("LaTeX ablation table saved.")
print(latex)


LaTeX ablation table saved.
\begin{table}[ht]
\centering
\caption{Ablation Study: Component Contribution to RLSH Performance}
\label{tab:ablation}
\begin{tabular}{lccc}
\toprule
\textbf{Model} & \textbf{Accuracy} & \textbf{Dir. Acc.} & \textbf{Coverage} \\
\midrule
Naive (always-hold) & ~0.500 & ~0.000 & 1.000 \\
LGBM-only (Phase 1, imbalanced) & ~0.560 & ~0.040 & 1.000 \\
RATAN-only (Huber, original) & ~0.530 & ~0.529 & 1.000 \\
RATAN-CE (retrained, CE loss) & 0.4250 & 0.7593 & 1.000 \\
LGBM-aug (RATAN attention-scaled) & 0.3514 & 0.5885 & 1.000 \\
Hybrid-no-gate (avg probs) & 0.4381 & 0.7636 & 1.000 \\
RLSH-full (RATAN+LGBM+Gate) & 0.4002 & 0.7489 & 1.0000 \\
\bottomrule
\end{tabular}
\end{table}


In [4]:
# ── Per-ticker RLSH accuracy bar chart ───────────────────────────────────────
CORE_TICKERS = ["SPY", "AAPL", "MSFT", "JPM", "GLD"]
ATTN_DIR = "../data/features/attention_weights"

per_ticker_dir = []
for t in CORE_TICKERS:
    b = abl["gating_per_ticker"][t]
    per_ticker_dir.append(b["dir_accuracy"])

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#1565C0","#2E7D32","#E65100","#7B1FA2","#F57F17"]
bars = ax.bar(CORE_TICKERS, per_ticker_dir, color=colors, edgecolor='white', linewidth=1.5)
ax.axhline(0.75, color='red', linestyle='--', linewidth=2, label='75% target')
ax.axhline(np.mean(per_ticker_dir), color='navy', linestyle=':', linewidth=1.5,
           label=f'Average: {np.mean(per_ticker_dir):.2%}')
for bar, val in zip(bars, per_ticker_dir):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05); ax.set_ylabel("Directional Accuracy", fontsize=12)
ax.set_title("RLSH-full: Per-Ticker Directional Accuracy (with confidence gate θ)",
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIAG_DIR}/per_ticker_accuracy.png", dpi=150, bbox_inches='tight')
plt.close()
print("Per-ticker accuracy chart saved.")


Per-ticker accuracy chart saved.


In [5]:
# ── Attention feature importance heatmap (SPY) ───────────────────────────────
ATTN_DIR = "../data/features/attention_weights"
alpha_spy = np.load(f"{ATTN_DIR}/alpha_SPY.npy")   # (N_test, 90)
mean_alpha = alpha_spy.mean(0)                       # (90,)

top_idx = mean_alpha.argsort()[-20:][::-1]
# Feature names from daily_features.csv
import pandas as pd
df_cols = pd.read_csv("../data/features/daily_features.csv", nrows=0).columns.tolist()
feat_names = [c for c in df_cols if c != "SPY_Target"]
top_names = [feat_names[i] if i < len(feat_names) else f"F{i}" for i in top_idx]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_names[::-1], mean_alpha[top_idx][::-1], color='steelblue', edgecolor='white')
ax.set_xlabel("Mean Attention Weight", fontsize=11)
ax.set_title("SPY: Top 20 Features by RATAN-CE Attention (averaged over test set)",
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DIAG_DIR}/attention_heatmap_SPY.png", dpi=150, bbox_inches='tight')
plt.close()
print("Attention heatmap saved.")
print(f"\nTop 5 features for SPY: {top_names[:5]}")


Attention heatmap saved.

Top 5 features for SPY: ['GLD_Open', 'GLD_High', 'JPM_Target', 'JPM_FFD_Price', 'DX_Vol_Local']


In [6]:
print("\n" + "="*60)
print("NB19 complete. Publication artifacts saved:")
import os
diag_dir = "../docs/diagrams"
for f in sorted(os.listdir(diag_dir)):
    path = os.path.join(diag_dir, f)
    size = os.path.getsize(path)
    print(f"  {f:40s}  {size/1024:.1f} KB")



NB19 complete. Publication artifacts saved:
  ablation_table.tex                        0.7 KB
  architecture.png                          59.5 KB
  attention_heatmap_SPY.png                 75.0 KB
  per_ticker_accuracy.png                   55.4 KB
  rlsh_architecture.pdf                     44.2 KB
  rlsh_architecture.png                     190.8 KB
  theta_sweep.png                           132.7 KB
